# Adaptive RAG — Route Every Query to the Right Strategy
## Route the Query — Local Index, Web Search, or Direct Answer?
⏱ ~15 min

**Adaptive RAG** is the pattern that routes each incoming query to the *cheapest retrieval strategy that will correctly answer it* — before any retrieval happens. A lightweight LLM classifier decides: does this question need direct generation, a private vectorstore lookup, or a live web search? The rest of the pipeline executes only the chosen path.

By the end of this workshop you will understand *why* a single fixed strategy wastes money, *how* a classifier makes routing decisions, and *how* to wire the three-way conditional graph in LangGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why adaptive routing exists; fixed vs. adaptive strategies |
| 2 | **Architecture** — Routing decision tree and LangGraph state |
| 3 | **Private Knowledge Base** — Build and inspect the vectorstore |
| 4 | **The Classifier** — `with_structured_output` and the `RouteDecision` schema |
| 5 | **Full Graph** — Wire classify → conditional edges → three answer nodes |
| 6 | **Observe and Debug** — Inspect routing decisions and classifier reasoning |
| 7 | **Extend** — Add a fourth strategy; tune the classifier prompt |
| * | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the repo requirements already installed)
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)
- `duckduckgo-search` for the web path (included in requirements)

### Key References
> Jeong, S., et al. (2024). *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity.* NAACL 2024. https://arxiv.org/abs/2403.14403
>
> Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> LangGraph conditional edges: https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges
>
> Companion examples: 17-corrective-rag (grades docs after retrieval), 27-self-rag (decides whether to retrieve), 26-rag-fusion (query expansion)

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "chromadb",
            "ddgs",
            # "duckduckgo-search",  # legacy package — langchain-community 0.3.31+ requires ddgs instead
            "langgraph",
            "pydantic",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
from typing import Literal, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid-looking: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Adaptive Routing Exists

---

### The Problem with a Fixed Strategy

Most RAG tutorials wire every query into the same pipeline: embed the query, search the vectorstore, stuff context into the prompt, call the LLM. That works for the happy path, but it wastes resources and degrades quality across query types:

| Query type | Fixed RAG result | What should happen |
|------------|-----------------|--------------------|
| "What is 7 x 8?" | Embeds, retrieves irrelevant chunks, calls LLM with noise in context | LLM answers directly — no retrieval needed |
| "What is our refund policy?" | Good — vectorstore has this | Vectorstore retrieval is correct |
| "What is Bitcoin's price today?" | Vectorstore returns stale or empty results; LLM hallucinates a number | Live web search is required |
| "Who wrote Hamlet?" | Retrieves unrelated company docs | LLM answers directly from world knowledge |

The core insight of Adaptive RAG (Jeong et al., 2024): **query complexity and retrieval need are independent axes.** A classifier trained on query characteristics can predict the right strategy before any retrieval occurs, saving both cost and latency.

---

### Three Retrieval Strategies Compared

| Strategy | When to use | Cost | Latency | Hallucination risk |
|----------|------------|------|---------|-------------------|
| **`simple`** — direct LLM | General knowledge, math, definitions, historical facts | Lowest (1 LLM call) | Fast | Low (LLM knows this) |
| **`vectorstore`** — private KB | Internal docs, policies, product specs, anything private | Low (embed + search + 1 LLM call) | Medium | Low if docs are accurate |
| **`web`** — live search | Current events, prices, recent news, real-time data | Medium (search API + 1 LLM call) | Slower | Medium (depends on sources) |

---

### Adaptive vs Fixed Retrieval

| Dimension | Fixed RAG | Adaptive RAG |
|-----------|-----------|-------------|
| Query analysis | None — all queries treated equally | Classifier runs before retrieval |
| Retrieval cost | Always paid | Paid only when needed |
| Live data support | No — vectorstore is static | Yes — web path available |
| General knowledge | LLM mixes retrieval noise into known answers | LLM answers cleanly without noise |
| Implementation complexity | Simple | Moderate (routing layer added) |

---

## Part 2 — Architecture: The Routing Decision Tree

---

### How the Graph Works

Adaptive RAG is a **two-phase** graph. Phase 1 classifies the query. Phase 2 executes exactly one of three answer strategies depending on the classification.

```
  User query (plain string)
         |
         v
  +------------------------------+
  |         classify             |  LLM with structured output
  |  RouteDecision(strategy=...) |  returns: 'simple' | 'vectorstore' | 'web'
  +--------------+---------------+
                 |
       +---------+-----------+
       |         |           |
  strategy=  strategy=  strategy=
  'simple' 'vectorstore'  'web'
       |         |           |
       v         v           v
  +---------+ +-----------+ +----------+
  | direct  | |vectorstore| |   web    |
  | answer  | |  answer   | |  answer  |
  |         | |           | |          |
  | No      | | Chroma    | |DuckDuckGo|
  |retrieval| | semantic  | | live     |
  | needed  | | search    | | search   |
  +----+----+ +-----+-----+ +----+-----+
       |            |            |
       +------------+------------+
                    |
                    v
                   END
              (answer in state)
```

> **Key design principle:** The classifier is the only node that runs on every query. The three answer nodes are mutually exclusive — only one executes per invocation. This is what makes the graph *adaptive* rather than just a sequential RAG chain.

---

### LangGraph State

The graph passes a single `TypedDict` between nodes. Each node reads what it needs and writes what it produces:

| Field | Type | Set by | Used by |
|-------|------|--------|---------|
| `question` | `str` | Caller at invocation | `classify`, all answer nodes |
| `strategy` | `str` | `classify` | `route()` conditional edge function |
| `context` | `str` | Answer nodes | Inspection / debugging |
| `answer` | `str` | Answer nodes | Caller reads final output |

---

### Structured Output: Why `with_structured_output`?

The classifier must return a value that is exactly one of `'simple'`, `'vectorstore'`, or `'web'`. Free-form text generation is unreliable here — the router function will fail if it receives `"I would use the vectorstore"` instead of `"vectorstore"`.

`llm.with_structured_output(RouteDecision)` instructs the model to emit JSON matching the Pydantic schema and validates the result before returning. If the model produces invalid JSON or a value outside `Literal['simple', 'vectorstore', 'web']`, Pydantic raises an error immediately rather than letting a bad value silently corrupt state downstream.

---

## Part 3 — Build the Private Knowledge Base

---

The `vectorstore` path answers questions about internal knowledge the LLM was never trained on: product policies, internal specs, private documentation. We build this in-memory using ChromaDB so the notebook is self-contained.

In a production system this would be a persistent ChromaDB or Pinecone collection loaded from your actual docs. The routing logic does not change — only the vectorstore backend.

In [ ]:
# ─── 3-A: Define private documents ───────────────────────────────────────────
# These represent internal knowledge that the base LLM does NOT know.
# The classifier should route any question about these topics to 'vectorstore'.

PRIVATE_DOCS = [
    Document(
        page_content="Our refund policy allows returns within 30 days of purchase with original receipt.",
        metadata={"source": "policy", "topic": "refunds"},
    ),
    Document(
        page_content="API rate limits are 1000 requests per minute for Pro plans and 100 for Free plans.",
        metadata={"source": "api-docs", "topic": "rate-limits"},
    ),
    Document(
        page_content="The platform supports SSO via SAML 2.0 and OAuth 2.0 providers including Google and GitHub.",
        metadata={"source": "security-docs", "topic": "authentication"},
    ),
    Document(
        page_content="Data is stored in EU-West-1 by default. US regions available on Enterprise plans.",
        metadata={"source": "infrastructure", "topic": "data-residency"},
    ),
    Document(
        page_content="Uptime SLA is 99.9% for Pro plans and 99.99% for Enterprise. Includes status page monitoring.",
        metadata={"source": "sla", "topic": "uptime"},
    ),
]

print(f"Private KB: {len(PRIVATE_DOCS)} documents")
for doc in PRIVATE_DOCS:
    print(f"  [{doc.metadata['topic']:18}] {doc.page_content[:70]}...")

In [ ]:
# ─── 3-B: Embed and index into ChromaDB ──────────────────────────────────────
# Using in-memory Chroma so no disk setup is needed.
# For production: pass persist_directory="./chroma_db" to Chroma.from_documents()

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=PRIVATE_DOCS,
    embedding=embeddings_model,
    collection_name="adaptive_rag_workshop",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vectorstore ready: {vectorstore._collection.count()} chunks indexed")

# Smoke test — verify retrieval works before wiring the graph
test_docs = retriever.invoke("refund policy")
print(f"\nSmoke test — 'refund policy' retrieves {len(test_docs)} doc(s):")
for d in test_docs:
    print(f"  [{d.metadata['topic']}] {d.page_content[:80]}")

In [ ]:
# ─── 3-C: Inspect similarity scores ──────────────────────────────────────────
# Good diagnostic: see what the vectorstore considers 'similar' for various queries.
# Queries that belong to the 'simple' or 'web' strategy should score LOW here.

test_queries = [
    ("What is your refund policy?", "vectorstore"),
    ("What is 7 multiplied by 8?", "simple"),
    ("What is Bitcoin's price today?", "web"),
    ("What are the API rate limits?", "vectorstore"),
]

print("Query vs vectorstore similarity (lower distance = more similar)")
print("-" * 70)

for query, expected in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=1)
    if results:
        doc, distance = results[0]
        similarity = 1 - distance  # ChromaDB uses cosine distance
        bar = "#" * int(similarity * 20)
        print(f"  {similarity:.3f} {bar:<20}  [{expected:<12}] {query}")
        print(f"           Best match: {doc.page_content[:60]}...")
    print()

---

## Part 4 — The Classifier: Structured Routing Decisions

---

The classifier is the heart of Adaptive RAG. It is a small LLM call that returns a validated `strategy` value. We use `with_structured_output` with a Pydantic model to guarantee the output is parseable and type-safe.

### Why a Pydantic schema?

```python
# Without structured output — fragile:
response = llm.invoke("classify: " + question)
strategy = response.content   # could be anything: "I think vectorstore", "vs", "VECTORSTORE"
                               # router function breaks on any of these

# With structured output — reliable:
decision = classifier.invoke([...])   # returns RouteDecision(strategy='vectorstore', reasoning='...')
strategy = decision.strategy          # always exactly 'simple' | 'vectorstore' | 'web'
```

The `reasoning` field is a bonus: it makes the classifier's logic inspectable, which is invaluable for debugging misroutes.

In [ ]:
# ─── 4-A: Define the routing schema and classifier ────────────────────────────

Strategy = Literal["simple", "vectorstore", "web"]


class RouteDecision(BaseModel):
    """Structured routing decision returned by the classifier."""

    strategy: Strategy
    reasoning: str  # The classifier explains its choice — invaluable for debugging


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# with_structured_output wraps the LLM to:
#   1. Inject the Pydantic schema into the system prompt / tool call
#   2. Parse the response JSON into a RouteDecision instance
#   3. Validate that strategy is exactly one of the three Literal values
classifier = llm.with_structured_output(RouteDecision)

CLASSIFIER_SYSTEM = (
    "Classify this question into one of three retrieval strategies:\n"
    "- 'simple': general knowledge the LLM knows well (math, history, science, definitions)\n"
    "- 'vectorstore': questions about private or internal knowledge "
    "(company policies, product specs, internal documentation)\n"
    "- 'web': questions requiring current or real-time information "
    "(news, prices, recent events, today's date)\n"
    "Choose the cheapest strategy that will answer the question correctly."
)

print("RouteDecision schema fields:", list(RouteDecision.model_fields.keys()))
print(f"Valid strategies: {Strategy.__args__}")

In [ ]:
# ─── 4-B: Run the classifier standalone — inspect routing decisions ────────────
# Before wiring the graph, test the classifier in isolation.
# This is the most important debugging step: verify routing is correct FIRST.

ANNOTATED_QUESTIONS = [
    ("What is 7 multiplied by 8?", "simple"),
    ("What is your refund policy?", "vectorstore"),
    ("What are the API rate limits for a Pro plan?", "vectorstore"),
    ("What is the current price of Bitcoin?", "web"),
    ("Who wrote Hamlet?", "simple"),
    ("What is the EU data residency option?", "vectorstore"),
    ("What happened in the news today?", "web"),
]

print(f"{'Question':<45} {'Expected':<12} {'Got':<12} {'Match'}")
print("-" * 80)

correct = 0
for question, expected_strategy in ANNOTATED_QUESTIONS:
    decision = classifier.invoke(
        [
            SystemMessage(CLASSIFIER_SYSTEM),
            HumanMessage(question),
        ]
    )
    match = decision.strategy == expected_strategy
    correct += int(match)
    print(
        f"{question[:44]:<45} {expected_strategy:<12} {decision.strategy:<12} {'OK' if match else 'MISS'}"
    )

print(f"\nClassifier accuracy: {correct}/{len(ANNOTATED_QUESTIONS)}")

In [ ]:
# ─── 4-C: Inspect classifier reasoning on edge cases ─────────────────────────
# The 'reasoning' field lets you understand WHY the classifier chose a strategy.
# Use this when debugging misroutes — often the prompt needs a small clarification.

edge_cases = [
    "What is our uptime guarantee?",        # could be 'simple' (common SLA) or 'vectorstore'
    "What programming languages exist?",    # borderline 'simple' vs 'web'
    "What is the weather today?",           # clearly 'web'
]

print("Classifier reasoning for edge cases:\n")
for question in edge_cases:
    decision = classifier.invoke(
        [
            SystemMessage(CLASSIFIER_SYSTEM),
            HumanMessage(question),
        ]
    )
    print(f"Q: {question}")
    print(f"   Strategy  : {decision.strategy}")
    print(f"   Reasoning : {decision.reasoning}")
    print()

---

## Part 5 — Full Graph: Wire the Three-Way Conditional

---

### LangGraph Node and Edge Anatomy

```python
graph.add_node("classify", classify)          # node name -> function
graph.add_edge(START, "classify")             # fixed edge: always goes here first
graph.add_conditional_edges(
    "classify",      # from this node
    route,           # call this function with the current state to decide
    {                # map function return values to node names
        "direct_answer":      "direct_answer",
        "vectorstore_answer": "vectorstore_answer",
        "web_answer":         "web_answer",
    },
)
```

The `route()` function reads `state['strategy']` (set by `classify`) and returns one of the three node names. LangGraph uses that return value to look up which node to execute next via the mapping dict.

### Answer Node Responsibilities

| Node | Retrieves from | What it puts in state |
|------|---------------|----------------------|
| `direct_answer` | Nothing | `context='none'`, `answer=LLM response` |
| `vectorstore_answer` | ChromaDB (top-k chunks) | `context=joined chunks`, `answer=LLM response` |
| `web_answer` | DuckDuckGo (top-3 results) | `context=search results`, `answer=LLM response` |

In [ ]:
# ─── 5-A: Define graph state and all node functions ───────────────────────────


class AdaptiveRAGState(TypedDict):
    question: str
    strategy: str    # filled by classify; read by route()
    context: str     # filled by answer nodes; useful for inspection
    answer: str      # final output; read by the caller


# DuckDuckGoSearchResults returns a list of result snippets; max_results controls cost
# Alternative simpler variant: DuckDuckGoSearchRun() — returns a single concatenated string
# web_search_tool = DuckDuckGoSearchRun()  # simpler, no structured output
search_tool = DuckDuckGoSearchResults(max_results=3)


def classify(state: AdaptiveRAGState) -> dict:
    """Route the query to the cheapest strategy that will answer correctly."""
    decision = classifier.invoke(
        [
            SystemMessage(CLASSIFIER_SYSTEM),
            HumanMessage(state["question"]),
        ]
    )
    # strategy value drives the conditional edge: "simple", "vectorstore", or "web"
    return {"strategy": decision.strategy}


def direct_answer(state: AdaptiveRAGState) -> dict:
    """Answer using LLM world knowledge — no retrieval."""
    response = llm.invoke(
        [
            SystemMessage("Answer the question directly and concisely."),
            HumanMessage(state["question"]),
        ]
    )
    return {"context": "none", "answer": response.content}


def vectorstore_answer(state: AdaptiveRAGState) -> dict:
    """Retrieve from private KB and answer using only the retrieved context."""
    docs = retriever.invoke(state["question"])
    context = "\n\n".join(d.page_content for d in docs)
    response = llm.invoke(
        [
            SystemMessage(
                "Answer using only the context below. "
                "If the answer is not in the context, say so.\n\nContext:\n" + context
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"context": context, "answer": response.content}


def web_answer(state: AdaptiveRAGState) -> dict:
    """Search the live web and answer using search results as context."""
    results = search_tool.invoke(state["question"])
    response = llm.invoke(
        [
            SystemMessage(
                "Answer using the web search results below.\n\nResults:\n" + str(results)
            ),
            HumanMessage(state["question"]),
        ]
    )
    # web results stored as context — the only source of facts for this path
    return {"context": str(results), "answer": response.content}


def route(
    state: AdaptiveRAGState,
) -> Literal["direct_answer", "vectorstore_answer", "web_answer"]:
    """Map the strategy string set by classify() to a node name."""
    return {
        "simple": "direct_answer",
        "vectorstore": "vectorstore_answer",
        "web": "web_answer",
    }[state["strategy"]]


print("Node functions defined:")
for fn in [classify, direct_answer, vectorstore_answer, web_answer, route]:
    print(f"  {fn.__name__}: {fn.__doc__}")

In [ ]:
# ─── 5-B: Build and compile the graph ─────────────────────────────────────────

graph = StateGraph(AdaptiveRAGState)

# Register nodes
graph.add_node("classify", classify)
graph.add_node("direct_answer", direct_answer)
graph.add_node("vectorstore_answer", vectorstore_answer)
graph.add_node("web_answer", web_answer)

# Fixed entry edge — classify always runs first
graph.add_edge(START, "classify")

# Conditional routing: route() reads state['strategy'] and returns a node name
graph.add_conditional_edges(
    "classify",
    route,
    {
        "direct_answer": "direct_answer",
        "vectorstore_answer": "vectorstore_answer",
        "web_answer": "web_answer",
    },
)

# All answer nodes converge at END
graph.add_edge("direct_answer", END)
graph.add_edge("vectorstore_answer", END)
graph.add_edge("web_answer", END)

app = graph.compile()

print("Graph compiled.")
print("Topology: START -> classify -> [direct_answer | vectorstore_answer | web_answer] -> END")

In [ ]:
# ─── 5-C: Run the full routing demo ───────────────────────────────────────────
# Expected strategies are annotated in comments.
# Compare the actual routing against your expectation from Part 4.

SAMPLE_QUESTIONS = [
    "What is 7 multiplied by 8?",             # simple      — LLM knows math
    "What is your refund policy?",            # vectorstore — private policy
    "What are the API rate limits for Pro?",  # vectorstore — private spec
    "What is the current price of Bitcoin?",  # web         — real-time data
    "Who wrote Hamlet?",                      # simple      — world knowledge
]

print(f"{'Question':<45} {'Strategy':<14} Answer preview")
print("=" * 100)

for question in SAMPLE_QUESTIONS:
    result = app.invoke(
        {"question": question, "strategy": "", "context": "", "answer": ""}
    )
    answer_preview = result["answer"].replace("\n", " ")[:55]
    print(f"{question:<45} [{result['strategy']:<12}] {answer_preview}...")

---

## Part 6 — Observe and Debug the Graph

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| Misroute: `simple` when should be `vectorstore` | LLM answers from training data; hallucination on private facts | Classifier prompt doesn't describe KB domain | Add specific topic keywords to the `'vectorstore'` description |
| Misroute: `web` when should be `vectorstore` | Web search runs but returns irrelevant external pages | Classifier doesn't know a private KB exists | Mention KB domain explicitly in the prompt |
| `vectorstore` path returns empty context | Docs exist but retrieval finds nothing relevant | Chunk content too sparse or embedding mismatch | Add richer docs; check similarity scores from Part 3-C |
| `web` path hallucinates despite results | Search returned content but answer ignores it | System prompt doesn't enforce grounding | Strengthen: "Use ONLY the search results below" |
| `route()` raises KeyError | `f"{strategy}_answer"` not in conditional edge map | Classifier returned unexpected value | Pydantic Literal prevents this — should never happen with structured output |

In [ ]:
# ─── 6-A: Verbose run — show full state at each node ─────────────────────────
# LangGraph's stream() yields intermediate state dictionaries as each node runs.
# This is the primary debugging tool: see exactly what each node writes to state.

debug_question = "What is the refund policy?"
initial_state = {"question": debug_question, "strategy": "", "context": "", "answer": ""}

print(f"Streaming execution for: '{debug_question}'\n")
print("-" * 60)

for step in app.stream(initial_state):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    print(f"[NODE: {node_name}]")
    for key, value in node_output.items():
        preview = str(value)[:100].replace("\n", " ")
        print(f"  {key:10} = {preview}")
    print()

In [ ]:
# ─── 6-B: Compare routing with reasoning captured in state ───────────────────
# Modify classify() to also return the classifier's reasoning.
# Useful for debugging edge cases where expected strategy differs from actual.


class AdaptiveRAGStateVerbose(TypedDict):
    question: str
    strategy: str
    reasoning: str   # NEW: classifier reasoning captured in state
    context: str
    answer: str


def classify_verbose(state: AdaptiveRAGStateVerbose) -> dict:
    decision = classifier.invoke(
        [
            SystemMessage(CLASSIFIER_SYSTEM),
            HumanMessage(state["question"]),
        ]
    )
    return {"strategy": decision.strategy, "reasoning": decision.reasoning}


def route_verbose(
    state: AdaptiveRAGStateVerbose,
) -> Literal["direct_answer", "vectorstore_answer", "web_answer"]:
    return {
        "simple": "direct_answer",
        "vectorstore": "vectorstore_answer",
        "web": "web_answer",
    }[state["strategy"]]


# Build a second graph that captures reasoning
graph_verbose = StateGraph(AdaptiveRAGStateVerbose)
graph_verbose.add_node("classify", classify_verbose)
graph_verbose.add_node("direct_answer", direct_answer)
graph_verbose.add_node("vectorstore_answer", vectorstore_answer)
graph_verbose.add_node("web_answer", web_answer)
graph_verbose.add_edge(START, "classify")
graph_verbose.add_conditional_edges(
    "classify",
    route_verbose,
    {
        "direct_answer": "direct_answer",
        "vectorstore_answer": "vectorstore_answer",
        "web_answer": "web_answer",
    },
)
graph_verbose.add_edge("direct_answer", END)
graph_verbose.add_edge("vectorstore_answer", END)
graph_verbose.add_edge("web_answer", END)
app_verbose = graph_verbose.compile()

# Run an edge case
edge_q = "What is our uptime guarantee?"
result = app_verbose.invoke(
    {"question": edge_q, "strategy": "", "reasoning": "", "context": "", "answer": ""}
)
print(f"Q: {edge_q}")
print(f"Strategy  : {result['strategy']}")
print(f"Reasoning : {result['reasoning']}")
print(f"Answer    : {result['answer'][:200]}")

---

## Part 7 — Extend: Add a Fourth Strategy

---

### What If You Need More Than Three Paths?

Real systems often need additional strategies:
- **`sql`** — query a relational database for structured / aggregate data
- **`hybrid`** — combine vectorstore + web for partial private / partial current
- **`code`** — run a code interpreter for computational questions
- **`human`** — escalate to a human agent for policy-sensitive questions

Adding a strategy requires four changes:
1. Add the new literal to the `Strategy` type
2. Write the new answer node function
3. Register the node and its `add_edge(..., END)` in the graph
4. Add the new key to the `add_conditional_edges` mapping dict

The classifier system prompt also needs updating to describe when to choose the new strategy.

Below is the pattern applied to add a `'database'` path that queries structured sales data:

In [ ]:
# ─── 7-A: Four-strategy graph extension ───────────────────────────────────────
# Adds a 'database' strategy for structured data questions.
# The DB query is mocked here — replace with real SQLAlchemy/SQLite logic.

# Step 1: Extend the Literal type
StrategyExtended = Literal["simple", "vectorstore", "web", "database"]


class RouteDecisionExtended(BaseModel):
    strategy: StrategyExtended
    reasoning: str


class AdaptiveRAGStateExtended(TypedDict):
    question: str
    strategy: str
    context: str
    answer: str


classifier_extended = llm.with_structured_output(RouteDecisionExtended)

CLASSIFIER_SYSTEM_V2 = (
    "Classify this question into one of four retrieval strategies:\n"
    "- 'simple': general knowledge the LLM knows well (math, history, definitions)\n"
    "- 'vectorstore': private/internal knowledge (policies, specs, internal docs)\n"
    "- 'web': current or real-time information (news, prices, recent events)\n"
    "- 'database': structured data questions (sales figures, counts, aggregates, reports)\n"
    "Choose the cheapest strategy that will answer correctly."
)


# Step 2: Write the new answer node
def database_answer(state: AdaptiveRAGStateExtended) -> dict:
    """Mock: query a structured database for analytics questions."""
    # In production: run a SQL query against your database
    mock_data = {
        "total revenue": "$4.2M in Q4 2024",
        "active users": "12,453 as of last week",
        "top product": "DataPulse Pro with 38% of revenue",
    }
    context = str(mock_data)
    response = llm.invoke(
        [
            SystemMessage(f"Answer using only these database results:\n{context}"),
            HumanMessage(state["question"]),
        ]
    )
    return {"context": context, "answer": response.content}


def classify_extended(state: AdaptiveRAGStateExtended) -> dict:
    decision = classifier_extended.invoke(
        [SystemMessage(CLASSIFIER_SYSTEM_V2), HumanMessage(state["question"])]
    )
    return {"strategy": decision.strategy}


def route_extended(
    state: AdaptiveRAGStateExtended,
) -> Literal["direct_answer", "vectorstore_answer", "web_answer", "database_answer"]:
    return {
        "simple": "direct_answer",
        "vectorstore": "vectorstore_answer",
        "web": "web_answer",
        "database": "database_answer",
    }[state["strategy"]]


# Steps 3 & 4: Build the four-node graph
graph_ext = StateGraph(AdaptiveRAGStateExtended)
graph_ext.add_node("classify", classify_extended)
graph_ext.add_node("direct_answer", direct_answer)
graph_ext.add_node("vectorstore_answer", vectorstore_answer)
graph_ext.add_node("web_answer", web_answer)
graph_ext.add_node("database_answer", database_answer)           # NEW
graph_ext.add_edge(START, "classify")
graph_ext.add_conditional_edges(
    "classify",
    route_extended,
    {
        "direct_answer": "direct_answer",
        "vectorstore_answer": "vectorstore_answer",
        "web_answer": "web_answer",
        "database_answer": "database_answer",                    # NEW
    },
)
graph_ext.add_edge("direct_answer", END)
graph_ext.add_edge("vectorstore_answer", END)
graph_ext.add_edge("web_answer", END)
graph_ext.add_edge("database_answer", END)                       # NEW
app_ext = graph_ext.compile()

# Test the extended graph
extended_questions = [
    "What was our total revenue last quarter?",   # database
    "What is 7 multiplied by 8?",                 # simple
    "What is the current Bitcoin price?",         # web
]

print("Extended graph (4 strategies):\n")
for q in extended_questions:
    result = app_ext.invoke({"question": q, "strategy": "", "context": "", "answer": ""})
    print(f"Q: {q}")
    print(f"   -> [{result['strategy']}] {result['answer'][:100]}...")
    print()

---

## Exercises

---

Work through these in order — each builds on the previous.

### Exercise 1 — Inspect the Reasoning Field

The `RouteDecision` schema has a `reasoning` field. Modify the original `app` (from Part 5) to capture and print this reasoning for every question in `SAMPLE_QUESTIONS`.

**Goal:** Verify the classifier's stated reasoning matches your own intuition. Find one question where the reasoning surprises you.

### Exercise 2 — Expand the Private Knowledge Base

Add three new documents to `PRIVATE_DOCS` (e.g., pricing tiers, support contacts, or product features). Write three matching questions that should route to `vectorstore` and confirm they do.

**Bonus:** Write an ambiguous question (e.g., "What is our enterprise uptime?") and observe which strategy wins. Does the classifier pick the private KB or the web?

### Exercise 3 — Break the Classifier and Fix It

Remove the `'vectorstore'` description from `CLASSIFIER_SYSTEM`. Run `SAMPLE_QUESTIONS`. Which questions now misroute? Restore the prompt with a more specific description and verify the fix.

**Goal:** Understand how the classifier prompt directly controls routing behavior. The classifier prompt is the most important tuning surface in Adaptive RAG.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Modify the graph's classify node to also return decision.reasoning in state.
# Print both strategy and reasoning for each question in SAMPLE_QUESTIONS.

# Hint: add 'reasoning': str to AdaptiveRAGState, return it from classify(),
# and read it from result['reasoning'] after app.invoke().

# TODO: add 'reasoning' field to a copy of AdaptiveRAGState
# TODO: modify classify() to return {"strategy": ..., "reasoning": ...}
# TODO: rebuild and compile the graph
# TODO: loop over SAMPLE_QUESTIONS and print strategy + reasoning for each

print("Exercise 1 — implement the reasoning capture above, then re-run")

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Extend PRIVATE_DOCS with at least 3 new documents.
# Write 3 matching questions and verify they route to 'vectorstore'.

extra_docs = [
    # TODO: add your documents here
    # Document(page_content="...", metadata={"source": "...", "topic": "..."}),
]

extra_questions = [
    # TODO: write questions whose answers live in extra_docs
]

# After filling in the above:
# 1. Rebuild the vectorstore with PRIVATE_DOCS + extra_docs
# 2. Recompile the graph with the new retriever
# 3. Run extra_questions and verify strategy == 'vectorstore' for each

print("Exercise 2 — add documents and questions above, then rebuild and test")

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
# Break the classifier by removing the 'vectorstore' description.
# Observe misroutes, then fix the prompt.

BROKEN_CLASSIFIER_SYSTEM = (
    "Classify this question into one of three retrieval strategies:\n"
    "- 'simple': general knowledge the LLM knows well\n"
    # 'vectorstore' description intentionally removed — observe what breaks
    "- 'web': current or real-time information\n"
    "Choose the cheapest strategy that will answer correctly."
)

broken_classifier = llm.with_structured_output(RouteDecision)

print("Running questions with broken classifier (no vectorstore description):\n")
for question, expected in ANNOTATED_QUESTIONS[:5]:
    decision = broken_classifier.invoke(
        [SystemMessage(BROKEN_CLASSIFIER_SYSTEM), HumanMessage(question)]
    )
    correct = decision.strategy == expected
    print(f"  {'OK  ' if correct else 'MISS'} [{decision.strategy:<12}] {question}")

print("\nTODO: note which questions misroute, then write a fixed BROKEN_CLASSIFIER_SYSTEM")

---

## Exercise Answer Key

Attempt the exercises before reading this section.

---

### Exercise 1 — Inspect the Reasoning Field

**What to add to state:**
```python
class AdaptiveRAGStateWithReasoning(TypedDict):
    question: str
    strategy: str
    reasoning: str   # add this field
    context: str
    answer: str
```

**What to return from `classify()`:**
```python
return {"strategy": decision.strategy, "reasoning": decision.reasoning}
```

**Expected output shape:**
```
Q: What is 7 multiplied by 8?
   Strategy  : simple
   Reasoning : This is a basic arithmetic question that the LLM can answer
               directly without any retrieval.
```

**Typical surprising finding:** `"What is our uptime guarantee?"` often routes to `simple` even though uptime data is in the KB — because the classifier's only knowledge of the KB comes from the prompt. If the prompt doesn't name the SLA/uptime domain, the classifier has no reason to prefer `vectorstore`. This is the key lesson: **describe the KB domain with concrete keywords in the classifier prompt**.

---

### Exercise 2 — Expand the Private Knowledge Base

**Example new documents:**
```python
extra_docs = [
    Document(page_content="Enterprise plans start at $999/month and include a dedicated Customer Success Manager."),
    Document(page_content="Customer support is available 24/7 via Slack Connect for Enterprise customers."),
    Document(page_content="The mobile app requires iOS 15 or higher and Android 10 or higher."),
]
```

**Matching questions:**
```python
extra_questions = [
    "How much does the Enterprise plan cost?",
    "How do I contact support as an Enterprise customer?",
    "What iOS version do I need for the mobile app?",
]
```

**Rebuild pattern:**
```python
all_docs = PRIVATE_DOCS + extra_docs
new_vectorstore = Chroma.from_documents(all_docs, embeddings_model, collection_name="adaptive_rag_v2")
new_retriever = new_vectorstore.as_retriever(search_kwargs={"k": 3})
# rebuild the graph using new_retriever inside vectorstore_answer
```

---

### Exercise 3 — Break the Classifier and Fix It

**Expected misroutes with `BROKEN_CLASSIFIER_SYSTEM`:**
- `"What is your refund policy?"` -> likely `simple` (LLM tries to answer from training data)
- `"What are the API rate limits?"` -> likely `simple` or `web`

**The fix — describe the KB with specific keywords:**
```python
FIXED_CLASSIFIER_SYSTEM = (
    "Classify this question into one of three retrieval strategies:\n"
    "- 'simple': general knowledge the LLM knows well (math, history, science, definitions)\n"
    "- 'vectorstore': questions about THIS COMPANY's private knowledge — "
    "refund policies, API rate limits, authentication, data residency, SLA, "
    "pricing, or any internal/product-specific information\n"
    "- 'web': current or real-time information (news, prices, recent events)\n"
    "Choose the cheapest strategy that will answer correctly."
)
```

**Key lesson:** The classifier has no direct knowledge of what is in the vectorstore. Its only signal is the system prompt. Vague descriptions like `"private knowledge"` are insufficient. Listing the specific domains and topics covered by the KB dramatically improves routing accuracy.

In [ ]:
# ── Answer Key: Exercise 1 sample solution ────────────────────────────────────
# app_verbose from Part 6-B already captures reasoning.
# Run this cell to see reasoning for all sample questions.

print("Reasoning for all sample questions:\n")
for question, expected in ANNOTATED_QUESTIONS:
    result = app_verbose.invoke(
        {"question": question, "strategy": "", "reasoning": "", "context": "", "answer": ""}
    )
    match = result["strategy"] == expected
    print(f"{'OK  ' if match else 'MISS'} [{result['strategy']:<12}] {question}")
    print(f"     Reasoning: {result['reasoning'][:120]}")
    print()

In [ ]:
# ── Answer Key: Exercise 2 sample solution ────────────────────────────────────

extra_docs_solution = [
    Document(
        page_content="Enterprise plans start at $999/month and include a dedicated Customer Success Manager.",
        metadata={"source": "pricing", "topic": "enterprise-pricing"},
    ),
    Document(
        page_content="Customer support is available 24/7 via Slack Connect for Enterprise customers.",
        metadata={"source": "support", "topic": "enterprise-support"},
    ),
    Document(
        page_content="The mobile app requires iOS 15 or higher and Android 10 or higher.",
        metadata={"source": "product", "topic": "mobile-requirements"},
    ),
]

all_docs_v2 = PRIVATE_DOCS + extra_docs_solution
vectorstore_v2 = Chroma.from_documents(
    all_docs_v2,
    embeddings_model,
    collection_name="adaptive_rag_v2",
)
retriever_v2 = vectorstore_v2.as_retriever(search_kwargs={"k": 3})

extra_questions_solution = [
    "How much does the Enterprise plan cost?",
    "How do I contact support as an Enterprise customer?",
    "What iOS version do I need for the mobile app?",
]

print("Exercise 2 — extended KB routing check:\n")
for q in extra_questions_solution:
    decision = classifier.invoke([SystemMessage(CLASSIFIER_SYSTEM), HumanMessage(q)])
    docs = retriever_v2.invoke(q)
    print(f"Q: {q}")
    print(f"   Strategy: {decision.strategy}  |  Top doc: {docs[0].page_content[:70]}")
    print()

---

## What's Next?

You now understand how to route queries adaptively before retrieval occurs. Here is where to go deeper:

### Grade documents after retrieval
- **Example 17 — Corrective RAG** (`../17-corrective-rag/`): Adds a grading step that evaluates whether each retrieved document is actually relevant to the query. If any chunk scores below threshold, the query is rewritten and retrieval is retried. Complements Adaptive RAG: this example decides *which strategy to use*; Corrective RAG decides *whether the retrieved docs are good enough*.

### Self-questioning retrieval
- **Example 27 — Self-RAG** (`../27-self-rag/`): Three LLM gates: (1) should I retrieve at all? (2) is each retrieved doc relevant? (3) is the generated answer grounded in the docs? The most thorough approach — but three times the LLM calls.

### Better recall via query expansion
- **Example 26 — RAG Fusion** (`../26-rag-fusion/`): Generates N paraphrased variants of the query in parallel, retrieves for each, then merges results using Reciprocal Rank Fusion (RRF). Improves recall on ambiguous or multi-faceted queries.

### Evaluate what you built
- **Example 16 — RAG Evaluation with RAGAS** (`../16-rag-eval-notebook/rag_eval_workbook.ipynb`): Score your pipeline on Faithfulness, Answer Relevance, and Context Recall using automated RAGAS metrics. Put numbers on the quality differences you observed during this workshop.

### Further reading
> Jeong, S., Baek, J., Cho, S., Hwang, S. J., & Park, J. C. (2024). *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity.* NAACL 2024. https://arxiv.org/abs/2403.14403
>
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> LangGraph — Conditional Edges: https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges
>
> LangChain — Structured Output: https://python.langchain.com/docs/concepts/structured_outputs/

---

*Workshop complete. Next: example 17 — Corrective RAG adds a document quality gate after the retrieval step.*